# 05 — طبقة الـ LLM والـ Artifacts

1. **LLM layer:** الـ factory، الـ fake provider، التعامل مع الأخطاء.
2. **Artifact contract:** الـ 7 أنواع (summary / chart / diagram / keywords / quiz / mcq / flashcards) لازم تطابق اللي الـ Frontend (Lovable) متوقعه — بنكتب validators من الـ contract ونجربها على: (أ) الـ backend وهو بيحفظ، (ب) اللي الـ chat بيولّده.
3. **Chat contract:** blocks / citations / mismatch block.

> الـ validators هنا مبنية على الـ contract اللي اتفقنا عليه (rendering facts من Lovable). لو الـ frontend اتغيّر، عدّل الـ validators.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
sys.path.insert(0, str(pathlib.Path.cwd() / "notebooks"))
from nb_utils import *
env = setup()
print(env)
rec = Recorder("05_llm_and_artifacts")

In [ ]:
import inspect, pkgutil, importlib, asyncio
api = get_api()
rows = []
try:
    import llm
    for m in [llm] + [importlib.import_module(x.name) for x in pkgutil.walk_packages(llm.__path__, "llm.")]:
        for n, o in inspect.getmembers(m):
            if not n.startswith("_") and getattr(o, "__module__", "") == m.__name__ and (inspect.isclass(o) or inspect.isfunction(o)):
                rows.append(dict(module=m.__name__, kind="class" if inspect.isclass(o) else "function", name=n))
except Exception as e:
    print("llm package problem:", type(e).__name__, e)
display(pd.DataFrame(rows))
rec.check("llm package imports and exposes classes/functions", bool(rows), f"{len(rows)} public objects", "llm")

## 1) LLM layer (unit-level)
بنحاول نلاقي الـ factory ونجرّب الـ fake client. لو الاسم مختلف عندك، اضبط `FACTORY` بإيدك.

In [ ]:
FACTORY = find_callable("llm", "get_llm", "get_client", "create", "factory", "build", "provider")
print("FACTORY ->", getattr(FACTORY, "__module__", None), getattr(FACTORY, "__name__", None))

def make_client():
    if FACTORY is None: raise Skip("factory not found - set FACTORY in this cell")
    return FACTORY() if not inspect.isclass(FACTORY) else FACTORY()

def generate(client, prompt):
    for n in ("generate", "complete", "chat", "invoke", "generate_text", "ask"):
        fn = getattr(client, n, None)
        if fn:
            res = fn(prompt)
            if inspect.isawaitable(res): res = asyncio.run(res)
            return res if isinstance(res, str) else " ".join(flatten_strings(res)) or str(res)
    raise Skip(f"{type(client).__name__} has no generate/complete/chat method - adapt generate()")

CLIENT = None
with rec.case("factory builds a client from settings (provider=fake)", "llm") as c:
    CLIENT = make_client(); c.note(type(CLIENT).__name__)
with rec.case("client returns a non-empty string for an English prompt", "llm"):
    need(CLIENT); assert generate(CLIENT, "Say hello in one sentence.").strip()
with rec.case("client handles an Arabic prompt", "llm"):
    need(CLIENT); assert generate(CLIENT, "قل مرحبا في جملة واحدة.").strip()
with rec.case("client handles a very long prompt (50k chars) without crashing", "llm") as c:
    need(CLIENT)
    try: generate(CLIENT, "word " * 10000); c.note("ok")
    except Skip: raise
    except Exception as e:
        raise Warn(f"raised {type(e).__name__}: {str(e)[:80]} - make sure the app maps this to a clean API error")
with rec.case("deterministic fake provider (same prompt -> same output)", "llm"):
    need(CLIENT); p = "What is 2+2?"; assert generate(CLIENT, p) == generate(CLIENT, p)

## 2) Artifact contract — validators
كل validator بيرجّع قائمة مشاكل: `error` (الـ frontend هيتكسر) أو `warn` (الشكل مش مثالي).

In [ ]:
def _num(x): return isinstance(x, (int, float)) and not isinstance(x, bool)

def v_summary(c):
    iss = []; b = c.get("body") if isinstance(c, dict) else None
    if not isinstance(b, str) or not b.strip(): return [("error", "body must be a non-empty markdown string")]
    for ln in b.splitlines():
        if re.match(r"#{4,}\s", ln): iss.append(("warn", f"heading deeper than ### is not rendered: {ln[:30]}"))
        if re.match(r"\s*\d+[.)]\s", ln): iss.append(("warn", "numbered lists are not rendered (use '- ')"))
        if ln.strip().startswith("|"): iss.append(("warn", "markdown tables are not rendered"))
        if ln.strip().startswith("```"): iss.append(("warn", "code fences are not rendered"))
    return iss

def v_chart(c):
    iss = []
    if c.get("chart_type") not in ("bar", "line", "pie", "scatter"): iss.append(("error", f"chart_type={c.get('chart_type')!r}"))
    data, series, xk = c.get("data"), c.get("series"), c.get("x_key")
    if not isinstance(xk, str): iss.append(("error", "x_key missing"))
    if not isinstance(data, list) or not data: return iss + [("error", "data must be a non-empty list")]
    if not isinstance(series, list) or not series: return iss + [("error", "series must be a non-empty list")]
    for s_ in series:
        k = s_.get("key") if isinstance(s_, dict) else None
        if not k: iss.append(("error", "each series needs a 'key'")); continue
        if not all(_num(row.get(k)) for row in data if isinstance(row, dict)): iss.append(("error", f"series '{k}' has non-numeric values"))
        col = s_.get("color")
        if col and not str(col).startswith("var(--"): iss.append(("warn", f"series color {col!r} should be a CSS var like var(--primary)"))
    if xk and not all(isinstance(r, dict) and xk in r for r in data): iss.append(("error", f"some rows lack x_key '{xk}'"))
    t = c.get("chart_type"); n = len(data)
    if t == "pie" and not 4 <= n <= 7: iss.append(("warn", f"pie has {n} slices (reads well with 4-7)"))
    if t != "pie" and not 10 <= n <= 30 and t in ("bar", "line"): iss.append(("warn", f"{n} points (reads well with 10-30)"))
    if t == "pie" and len(series) > 1: iss.append(("warn", "pie uses only series[0]"))
    if t == "scatter": iss.append(("warn", "scatter is not bound in the frontend yet"))
    return iss

def v_diagram(c):
    iss, nodes, edges = [], c.get("nodes"), c.get("edges")
    if not isinstance(nodes, list) or not nodes: return [("error", "nodes must be a non-empty list")]
    ids = set()
    for n in nodes:
        if not isinstance(n, dict) or "id" not in n: iss.append(("error", "node without id")); continue
        ids.add(str(n["id"]))
        if not (_num(n.get("x")) and _num(n.get("y"))): iss.append(("error", f"node {n['id']} needs numeric x,y (no auto-layout)"))
    for e in edges or []:
        if str(e.get("source")) not in ids or str(e.get("target")) not in ids: iss.append(("error", f"edge {e} references a missing node"))
    if not 4 <= len(nodes) <= 12: iss.append(("warn", f"{len(nodes)} nodes (reads well with 4-12)"))
    return iss

def v_keywords(c):
    items = c.get("items")
    if not isinstance(items, list) or not items: return [("error", "items must be a non-empty list")]
    iss = [("error", f"bad item {i}") for i in items if not (isinstance(i, dict) and isinstance(i.get("term"), str) and _num(i.get("weight")))]
    if not 10 <= len(items) <= 30: iss.append(("warn", f"{len(items)} keywords (reads well with 10-30)"))
    return iss

def v_quiz(c):
    q = c.get("questions")
    if not isinstance(q, list) or not q: return [("error", "questions must be a non-empty list")]
    iss = [("error", f"question needs id/question/answer: {str(x)[:50]}") for x in q if not (isinstance(x, dict) and {"id", "question", "answer"} <= set(x))]
    if not 3 <= len(q) <= 8: iss.append(("warn", f"{len(q)} questions (3-8 recommended)"))
    return iss

def v_mcq(c):
    q = c.get("questions")
    if not isinstance(q, list) or not q: return [("error", "questions must be a non-empty list")]
    iss = []
    for x in q:
        o, ci = x.get("options") if isinstance(x, dict) else None, x.get("correct_index") if isinstance(x, dict) else None
        if not (isinstance(o, list) and len(o) >= 2): iss.append(("error", "options must have >= 2 entries"))
        elif not (isinstance(ci, int) and 0 <= ci < len(o)): iss.append(("error", f"correct_index {ci!r} out of range"))
        elif len({str(i) for i in o}) != len(o): iss.append(("warn", "duplicate options"))
    if not 3 <= len(q) <= 8: iss.append(("warn", f"{len(q)} questions (3-8 recommended)"))
    return iss

def v_flash(c):
    cards = c.get("cards")
    if not isinstance(cards, list) or not cards: return [("error", "cards must be a non-empty list")]
    iss = [("error", "card needs front/back") for x in cards if not (isinstance(x, dict) and x.get("front") and x.get("back"))]
    if not 4 <= len(cards) <= 15: iss.append(("warn", f"{len(cards)} cards (4-15 recommended)"))
    return iss

VALIDATORS = dict(summary=v_summary, chart=v_chart, diagram=v_diagram, keywords=v_keywords, quiz=v_quiz, mcq=v_mcq, flashcards=v_flash)

GOOD = {
    "summary": {"body": "# Title\n## Overview\n- point one\n- **bold** point with `code`\n"},
    "chart": {"chart_type": "bar", "x_key": "month", "data": [{"month": f"m{i}", "sales": i * 3} for i in range(12)],
              "series": [{"key": "sales", "label": "Sales", "color": "var(--primary)"}]},
    "diagram": {"nodes": [{"id": str(i), "label": f"N{i}", "x": i * 120, "y": (i % 2) * 90} for i in range(6)],
                "edges": [{"id": f"e{i}", "source": str(i), "target": str(i + 1)} for i in range(5)]},
    "keywords": {"items": [{"term": f"term{i}", "weight": round(1 - i / 20, 2)} for i in range(12)]},
    "quiz": {"questions": [{"id": f"q{i}", "question": f"Question {i}?", "answer": f"Answer {i}"} for i in range(4)]},
    "mcq": {"questions": [{"id": f"q{i}", "question": f"Question {i}?", "options": ["a", "b", "c", "d"], "correct_index": i % 4} for i in range(4)]},
    "flashcards": {"cards": [{"front": f"Front {i}", "back": f"Back {i}"} for i in range(6)]},
}
BAD = {
    "summary": {"body": ""}, "chart": {"chart_type": "donut", "x_key": "a", "data": [], "series": []},
    "diagram": {"nodes": [{"id": "1", "label": "no coords"}], "edges": [{"source": "1", "target": "ghost"}]},
    "keywords": {"items": [{"term": "x"}]}, "quiz": {"questions": [{"question": "no id"}]},
    "mcq": {"questions": [{"options": ["a", "b"], "correct_index": 5}]}, "flashcards": {"cards": [{"front": "only front"}]},
}
for t, v in VALIDATORS.items():
    with rec.case(f"validator self-test: '{t}' accepts a good sample and rejects a bad one", "contract"):
        good_err = [m for lvl, m in v(GOOD[t]) if lvl == "error"]
        bad_err = [m for lvl, m in v(BAD[t]) if lvl == "error"]
        assert not good_err, f"good sample rejected: {good_err}"
        assert bad_err, "bad sample was NOT rejected -> validator too weak"

## 3) الـ Backend وهو بيحفظ Artifact (Save as artifact)

In [ ]:
U = api.new_user("art"); WS = api.make_workspace(U); PID = api.make_project(U, WS)
SAVED = {}
def create_artifact(t, content, scope=None, title=None):
    m, p = api.path("artifact_create")
    body = PAYLOADS["artifact"](t, scope or {"project_id": PID}, title or f"nb-{t}", content)
    if "{" in p: return api.call(m, fill(p, PID), token=U.token, json=body)
    return api.call(m, p, token=U.token, json=body)

for t in VALIDATORS:
    with rec.case(f"save valid '{t}' artifact -> 2xx and round-trips unchanged", "artifact-api") as c:
        r = create_artifact(t, GOOD[t]); assert r.status_code in (200, 201), f"HTTP {r.status_code}: {r.text[:200]}"
        js = unwrap(safe_json(r)); SAVED[t] = get_id(r)
        assert find_key(js, "content") == GOOD[t], "stored content differs from what was sent"
for t in VALIDATORS:
    with rec.case(f"backend rejects an INVALID '{t}' artifact (4xx)", "artifact-api") as c:
        r = create_artifact(t, BAD[t]); c.note(f"HTTP {r.status_code}")
        assert r.status_code < 500, f"server error {r.status_code}"
        if r.status_code < 300: raise Warn("invalid content was accepted -> frontend may crash when rendering it")
with rec.case("unknown artifact_type is rejected", "artifact-api"):
    r = create_artifact("hologram", {"x": 1}); assert 400 <= r.status_code < 500, f"HTTP {r.status_code}"
with rec.case("artifact list returns the saved artifacts", "artifact-api") as c:
    need(SAVED)
    m, p = api.path("artifact_list")
    r = api.call(m, fill(p, PID) if "{" in p else p, token=U.token, params={} if "{" in p else {"project_id": PID})
    assert r.status_code == 200, f"HTTP {r.status_code}"
    c.note(f"{len(find_hits(safe_json(r)))} items")

## 4) توليد الـ Artifacts من الـ Chat
بنرفع ملف، نسأل مع `requested_type` لكل نوع، ونطبّق الـ validator على اللي رجع.

In [ ]:
r = api.upload(U, PID, "orion_manual.txt", make_corpus()["orion_manual.txt"][0], "text/plain")
st, secs = api.wait_ready(U, get_id(r)); print("ingestion:", st, f"{secs:.1f}s")
CID = api.make_conversation(U, PID)

def find_artifacts(obj, out=None):
    """Collect dicts that look like an artifact block: have 'content' and an artifact type."""
    out = [] if out is None else out
    if isinstance(obj, dict):
        t = obj.get("artifact_type") or (obj.get("type") if obj.get("type") in VALIDATORS else None)
        if t and isinstance(obj.get("content"), dict): out.append((t, obj["content"]))
        for v in obj.values(): find_artifacts(v, out)
    elif isinstance(obj, list):
        for v in obj: find_artifacts(v, out)
    return out

GEN = []
for t in VALIDATORS:
    with rec.case(f"chat generates a valid '{t}' artifact", "artifact-gen") as c:
        need(st == "READY" or None)
        t0 = time.perf_counter(); r = api.send_message(U, CID, f"Create a {t} about the Orion-7 reactor.", requested_type=t); dt = time.perf_counter() - t0
        assert r.status_code in (200, 201), f"HTTP {r.status_code}: {r.text[:200]}"
        arts = [a for a in find_artifacts(safe_json(r)) if a[0] == t]
        mism = find_key(safe_json(r), "mismatch") or ("mismatch" in r.text and "requested" in r.text)
        if not arts:
            GEN.append(dict(type=t, secs=round(dt, 2), errors=0, warnings=0, note="no artifact block" + (" (mismatch card returned)" if mism else "")))
            raise AssertionError("no artifact block in the assistant message" + (" - a mismatch card was returned instead" if mism else ""))
        iss = VALIDATORS[t](arts[0][1])
        errs, warns = [m for l, m in iss if l == "error"], [m for l, m in iss if l == "warn"]
        GEN.append(dict(type=t, secs=round(dt, 2), errors=len(errs), warnings=len(warns), note="; ".join(errs + warns)[:120]))
        assert not errs, "; ".join(errs)
        if warns: raise Warn("; ".join(warns))
if GEN: display(pd.DataFrame(GEN))

## 5) Chat blocks و الـ Mismatch

In [ ]:
blocks = am = None
with rec.case("assistant message exposes blocks[] (text|code|artifact|mismatch)", "chat-contract") as c:
    r = api.send_message(U, CID, "Summarize the maintenance schedule."); assert r.status_code in (200, 201), f"HTTP {r.status_code}"
    am = unwrap(safe_json(r)); am = am.get("assistant_message", am) if isinstance(am, dict) else am
    blocks = am.get("blocks") if isinstance(am, dict) else None
    assert isinstance(blocks, list) and blocks, "no blocks[] in assistant_message"
    kinds = {b.get("type") for b in blocks if isinstance(b, dict)}
    assert kinds <= {"text", "code", "artifact", "mismatch"}, f"unknown block types: {kinds}"
    c.note(sorted(kinds))

with rec.case("citation markers [n] are valid and in range", "chat-contract") as c:
    need(blocks, am)
    txt = " ".join(flatten_strings([b for b in blocks if isinstance(b, dict) and b.get("type") == "text"]))
    cits = am.get("citations") or []
    marks = {int(x) for x in re.findall(r"\[(\d+)\]", txt)}
    if not marks: raise Warn("no [n] markers in the text")
    assert max(marks) <= len(cits), f"[{max(marks)}] used but only {len(cits)} citations"
    assert all(isinstance(x, dict) and {"chunk_id"} <= set(x) for x in cits), "citation objects need id/message_id/chunk_id"

In [ ]:
def all_dicts(o):
    if isinstance(o, dict):
        yield o
        for v in o.values(): yield from all_dicts(v)
    elif isinstance(o, list):
        for v in o: yield from all_dicts(v)

with rec.case("mismatch block (when returned) has requested / reason / alternatives[]", "chat-contract") as c:
    r = api.send_message(U, CID, "Make a chart of the reactor's maintenance policy.", requested_type="chart")
    assert r.status_code in (200, 201), f"HTTP {r.status_code}"
    ds = list(all_dicts(safe_json(r)))
    mm = [d for d in ds if d.get("type") == "mismatch"]
    arts = [a for a in find_artifacts(safe_json(r)) if a[0] == "chart"]
    if not mm and not arts: raise AssertionError("neither a mismatch block nor a chart artifact was returned")
    if mm:
        assert {"requested", "reason", "alternatives"} <= set(mm[0]), f"mismatch block keys: {sorted(mm[0])}"
        assert isinstance(mm[0]["alternatives"], list), "alternatives must be a list"
        c.note("mismatch block returned")
    else:
        c.note("a chart artifact was generated instead of a mismatch")

In [ ]:
rec.summary(); rec.table(); rec.save()